<a href="https://colab.research.google.com/github/mugalan/classical-mechanics-from-a-geometric-point-of-view/blob/main/mechanics/answers-to-selected-assignments/Answers_to_Assignment_on_Free_Rigid_Body_Motion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
import numpy as np
import scipy as sp
from scipy.integrate import odeint
import math
from numpy import linalg
import sympy

from sympy import symbols
from sympy import *
from sympy.physics.mechanics import dynamicsymbols, init_vprinting
init_vprinting(pretty_print=True)

import plotly.graph_objects as go

In [2]:
# Install the simulation helpers
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"
import sims
mr = sims.RigidBodySim()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# Assignment on Free Rigid Body Motion

Consider a rigid body with no external forces or
force moments. Let $\mathbf e$ be an inertial frame,
and let $\mathbf b=\mathbf eR$ be a principal-axis
body frame with origin at the center of mass.
The inertia tensor in $\mathbf b$ is
$$
\mathbb{I}=
\operatorname{diag}(2,3,4)\,\mathrm{kg\,m^2}.
$$
Let $\Pi=\mathbb{I}\Omega$ be the body representation
of angular momentum about the center of mass. Answer the following:

* Write down the rotational equations of motion
using $(R,\Pi)$ as the variables. Show that
$$
\pi=R\Pi,\qquad
E=\frac12\Pi^T\mathbb{I}^{-1}\Pi,\qquad
h^2=\Pi^T\Pi
$$
are conserved.

* Identify the steady rotations about the three
principal axes.

* Plot the intersection of the body momentum sphere $\|\Pi(t)\|=h$ and the the Kinetic energy ellipsoid $E=\frac12\Pi^T\mathbb{I}^{-1}\Pi$

* Linearize the body angular momentum
equations about each nonzero steady rotation.
Determine which rotations have oscillatory transverse
perturbations and which have exponentially growing
transverse perturbations.

* Simulate the rotational motion
with $R(0)=I_{3\times3}$
$$
\Omega(0)=\Omega_0
\begin{bmatrix}1\\\varepsilon\\\varepsilon\end{bmatrix},
\qquad
\Omega(0)=\Omega_0
\begin{bmatrix}\varepsilon\\1\\\varepsilon\end{bmatrix},
\qquad
\Omega(0)=\Omega_0
\begin{bmatrix}\varepsilon\\\varepsilon\\1\end{bmatrix},
$$
where $\Omega_0=1\,\mathrm{s^{-1}}$ and
$\varepsilon=0.01$. Use $\Pi(0)=\mathbb{I}\Omega(0)$
and simulate for at least $60\,\mathrm{s}$.

* Plot the components of $\Omega(t)$ and $\Pi(t)$ for the above conditions.
Compare the behavior with your linearized analysis.

* For each simulation, plot the path of $\Pi(t)$
on the angular momentum sphere and display the
corresponding energy ellipsoid.
Also plot the spatial motion of the body axis
initially closest to the angular velocity direction.
Explain the difference between a steady body
angular momentum and a stationary body orientation.

* Monitor the errors in $E$, $\|\Pi\|$, and
$\pi=R\Pi$, together with the orthogonality error
$\|R^TR-I_{3\times3}\|_F$.
Check that your conclusions persist when the
numerical tolerances or time step are refined.


#Sample Answers

## Equations of motion in the spatial frame

\begin{align*}
\dot{R}
&=\widehat{\omega}R,
\qquad \omega=\mathbb{I}_R^{-1}\pi\quad \mathbb{I}_R=R\mathbb I R^T,\\
\dot{\pi}
&=0
\end{align*}


From this it follows that:

* The spatial angular momentum $\pi(t)=\mu$ is constant.
* Since
$$\dfrac{d}{dt}KE=T_e^T\Omega=0$$
we see that the total kinetic energy is constant; $KE=E=\mathrm{constant}$.
* Since $\|\Pi(t)\|=\|\pi(t)\|=\|\mu\|=h=\mathrm{constant}$

## Equations of motion in the body frame

\begin{align*}
\dot{R}
&=R\,\widehat{\mathbb{I}^{-1}\Pi},\\
\dot{\Pi}
&=\Pi\times(\mathbb{I}^{-1}\Pi).
\end{align*}


When $\mathbb{I}$ is diagonal, these equations admit the equilibrium solutions
$$
\Pi(t)\equiv\pm h
\begin{bmatrix}1\\0\\0\end{bmatrix},
\qquad
\Pi(t)\equiv\pm h
\begin{bmatrix}0\\1\\0\end{bmatrix},
\qquad
\Pi(t)\equiv\pm h
\begin{bmatrix}0\\0\\1\end{bmatrix}.
$$
Since $\Omega=\mathbb{I}^{-1}\Pi$, these solutions
correspond to steady rotations about the principal
axes. They are equilibria of the body angular momentum
equations, but the orientation $R$ continues to change.
They are therefore called *relative equilibria*
of the rigid body motion. When the principal moments
are distinct, these are the only equilibria on
$\|\Pi\|=h$.

### Plotting the intersection of the Angular Momentum sphere with the Kinetic Energy ellipsoid

In [24]:
#### Defining the parameters
M=10;l=2;w=1;h=4; II_1=M*(w**2+h**2)/12; II_2=M*(l**2+h**2)/12;II_3=M*(w**2+l**2)/12;
cubeDimensions={'l':l,'w':w,'h':h,'xp':l,'yp':w,'zp':h,};
parameters={'CM':np.array([l/2-cubeDimensions['xp'],w/2-cubeDimensions['yp'],h/2-cubeDimensions['zp']]), 'g':10, 'M':M, 'II':np.array([[II_1,0.,0.],[0.,II_2,0.],[0.,0.,II_3]])};

In [26]:
#### Definig the initial conditions
ICq=mr.q_from_axis_angles(0*np.pi/18,np.array([1,0,0])); ICR=mr.r_from_quaternions(ICq);
ICOmega=np.array([1.,1.,5.]); ICo=np.array([0.,0.,0.]); ICdoto=np.array([0.,0.,0.]);
ICXC=np.array([0,0,0]);
ICs=[[ICR,ICo],ICR@ICOmega,ICdoto,ICXC]

In [27]:
II=parameters['II'];
magPi=linalg.norm(II@ICOmega);
KE=0.5*ICOmega.T@II@ICOmega;
spatialpi=ICR@II@ICOmega;

In [28]:
fig, info = mr.plot_momentum_energy_intersection(
    parameters={"II": II},
    ICOmega=ICOmega,
    ICR=ICq,                 # or None
    overlay_spatial=False,    # set True to overlay rotated surfaces
    grid_phi=60,
    grid_theta=120,
    tol=2e-3,
    title="Polhode in body π-space"
)

## Numerical Verification

### Simulating the dynamics

#### Defining the force model

In [29]:
def externalForceModel(qq,parameters,X):
  #Heavy Top Object
  M=parameters['M']; II=parameters['II']; g=parameters['g']; CM=parameters['CM'];
  R=X[0][0]; omega=X[1]; spi=R@ II@ R.T @ omega
  fe=np.array([0,0,0]);
  taue=np.array([0,0,0]);
  return [taue,fe]

def actuator(qq,parameters, t, X, taue,fe):
  tauu=np.array([0,0,0]);
  fu=np.array([0,0,0]);
  return [tauu,fu]
mr.set_external_force_model(externalForceModel)
mr.set_actuator(actuator)

#### Simulating the dynamics

In [30]:
dt=0.01; Tmax=5.;
aa4=mr.simulating_a_cube(dt, Tmax, cubeDimensions, parameters,ICs)
fig1=mr.animated_cube_flat_shading(aa4,'Cube Pivoted at a point')

### Plotting the nutation angle

In [9]:
dt=0.1; Tmax=10;
Xs=mr.runga_kutta_method(dt,Tmax,parameters,ICs);

In [10]:
II=parameters['II'];

In [11]:
alphat=[];
for X in Xs:
  R=X[0][0];
  Pi=R@II@R.T@X[1];
  alphat+=[round(math.acos(Pi[2]/linalg.norm(Pi)), 5)]
t=np.linspace(0,Tmax+dt,round(Tmax/dt)+2)

In [12]:
fig = go.Figure(data=go.Scatter(x=t, y=alphat, mode='markers'))
fig.show()